In [1]:
! pip install azure-ai-projects azure-monitor-opentelemetry opentelemetry-instrumentation-openai-v2

In [2]:
import pkg_resources
print([d.project_name for d in pkg_resources.working_set if "openai" in d.project_name.lower()])


['openai', 'opentelemetry-instrumentation-openai-v2']


/tmp/ipykernel_39346/2353942542.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
! pip install --upgrade pip


In [4]:
! pip install opentelemetry-instrumentation-openai-v2


In [5]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
     credential=DefaultAzureCredential(),
     endpoint="https://sarath-8734-resource.services.ai.azure.com/api/projects/sarath-8734",
)
connection_string = project_client.telemetry.get_application_insights_connection_string()

In [6]:
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m ensurepip --upgrade
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m pip install --upgrade pip
! /workspaces/Azure-ai-foundry01/.venv/bin/python -m pip install opentelemetry-instrumentation-openai-v2
# Restart is needed


Looking in links: /tmp/tmpry8fzaj5


In [7]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

configure_azure_monitor(connection_string=connection_string)
OpenAIInstrumentor().instrument()

In [8]:

client = project_client.get_openai_client(
     api_version="2024-10-21"  # Use correct API version for your environment
)
response = client.chat.completions.create(
     model="gpt-4o", 
     messages=[{"role": "user", "content": "Write a short poem on open telemetry."}],
)
print(response.choices[0].message.content)

/workspaces/Azure-ai-foundry01/.venv/lib/python3.12/site-packages/opentelemetry/sdk/_events/__init__.py:53: LogDeprecatedInitWarning: LogRecord init with `trace_id`, `span_id`, and/or `trace_flags` is deprecated since 1.35.0. Use `context` instead.
  log_record = LogRecord(


**OpenTelemetry's Melody**  

Through the threads of code, it weaves,  
Observing all, as data breathes.  
Traces, metrics, logs align,  
Guiding truth through every line.  

Across the stacks and spans it flows,  
Unveiling secrets no one knows.  
A single source, a beacon bright,  
Unifying in its light.  

From clouds above to code below,  
It maps the paths we long to know.  
With OpenTelemetry in hand,  
The chaos bends to what we understand.  


In [9]:
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

In [10]:
def build_prompt_with_context(claim: str, context: str) -> str:
    return [{'role': 'system', 'content': "I will ask you to assess whether a particular scientific claim, based on evidence provided. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence."},
            {'role': 'user', 'content': f"""
                The evidence is the following: {context}

                Assess the following claim on the basis of the evidence. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence. Do not output any other text.

                Claim:
                {claim}

                Assessment:
            """}]

@tracer.start_as_current_span("assess_claims_with_context")
def assess_claims_with_context(claims, contexts):
    responses = []
    for claim, context in zip(claims, contexts):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=build_prompt_with_context(claim=claim, context=context),
        )
        responses.append(response.choices[0].message.content.strip('., '))

    return responses

In [11]:
claims = [
    "Drinking water improves memory.",
    "The Earth is flat.",
    "Vitamin C prevents the common cold."
]

contexts = [
    "A recent study showed that hydration improves cognitive performance.",
    "Scientific consensus and satellite imagery confirm the Earth is spherical.",
    "Multiple clinical trials have found no significant effect of Vitamin C in preventing colds."
]

# Call your function
assessments = assess_claims_with_context(claims, contexts)

print(assessments)


/workspaces/Azure-ai-foundry01/.venv/lib/python3.12/site-packages/opentelemetry/sdk/_events/__init__.py:53: LogDeprecatedInitWarning: LogRecord init with `trace_id`, `span_id`, and/or `trace_flags` is deprecated since 1.35.0. Use `context` instead.
  log_record = LogRecord(


['NEE', 'False', 'False']
